# 04 - Feature Engineering

Tạo hai bài toán:

- **Day 0**: dự báo ngay từ thời điểm issue được tạo.
- **Day 7 landmark**: chỉ xét issue vẫn còn tồn tại sau 7 ngày; dùng hoạt động xảy ra trong 7 ngày đầu.

Mặc định `strict_no_leakage: true`, nên Day-0 **không dùng** các field snapshot như final status, resolution, full-lifetime comments, v.v.

In [1]:
from pathlib import Path
import sys, yaml, pandas as pd, numpy as np

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

with open(ROOT / "configs" / "experiment.yaml", "r", encoding="utf-8") as f:
    CFG = yaml.safe_load(f)

print("ROOT =", ROOT)
print("random_seed =", CFG["random_seed"])

ROOT = d:\VNUK\Eureka 2026\Eureka_2026
random_seed = 42


In [2]:
from src.features import build_day0_table, build_day7_table, feature_spec
from src.quality import summarize_missing, assert_no_forbidden_predictors

cohort_path = ROOT / "data" / "processed" / "cohort.parquet"
cohort = pd.read_parquet(cohort_path)

print("Cohort:", cohort.shape)
cohort.head()

Cohort: (835643, 25)


,repository,project,issue_key,created,updated,resolution_date,initial_priority,initial_issue_type,initial_status,initial_assigned,...,status_changes_7d,assignee_changes_7d,priority_changes_7d,comments_7d,inactivity_days_at_7d,project_cutoff,event,duration_days,created_weekday,created_hour
0,Apache,FLEX,FLEX-35423,2021-11-23 11:45:31+00:00,2021-11-23 11:46:42+00:00,2021-11-23 11:46:42+00:00,Critical,Task,Open,0,...,1,0,0,0,6.999169,2021-11-23 11:46:42+00:00,True,0.000822,1,11
1,Apache,FLEX,FLEX-35422,2021-09-20 17:55:30+00:00,2021-09-20 17:55:30+00:00,NaT,Major,Task,Open,0,...,0,0,0,0,7.000000,2021-11-23 11:46:42+00:00,False,63.743889,0,17
2,Apache,FLEX,FLEX-35416,2021-03-24 01:12:07+00:00,2021-03-24 01:12:51+00:00,NaT,Major,Bug,Open,0,...,0,0,0,0,6.999485,2021-11-23 11:46:42+00:00,False,244.440683,2,1
3,Apache,FLEX,FLEX-35415,2021-03-21 10:39:15+00:00,2021-03-21 11:24:00+00:00,NaT,Minor,Improvement,Open,0,...,0,0,0,0,6.968914,2021-11-23 11:46:42+00:00,False,247.046840,6,10
4,Apache,FLEX,FLEX-35413,2020-10-07 17:54:01+00:00,2020-12-07 15:14:46+00:00,2020-12-07 15:14:46+00:00,Trivial,Bug,Open,0,...,2,0,0,0,6.999026,2021-11-23 11:46:42+00:00,True,60.889410,2,17


In [3]:
strict = bool(CFG["features"]["strict_no_leakage"])
landmark_days = int(CFG["features"]["landmark_days"])

day0 = build_day0_table(
    cohort,
    strict_no_leakage=strict,
)

day7 = build_day7_table(
    cohort,
    landmark_days=landmark_days,
    strict_no_leakage=strict,
)

print("Day0:", day0.shape)
print("Day7:", day7.shape)

Day0: (835643, 10)
Day7: (420029, 16)


In [4]:
num0, cat0 = feature_spec("day0", strict_no_leakage=strict)
num7, cat7 = feature_spec(
    "day7",
    strict_no_leakage=strict,
    landmark_days=landmark_days,
)

assert_no_forbidden_predictors(num0 + cat0)
assert_no_forbidden_predictors(num7 + cat7)

print("Day0 numerical:", num0)
print("Day0 categorical:", cat0)
print("Day7 numerical:", num7)
print("Day7 categorical:", cat7)

Day0 numerical: ['created_weekday', 'created_hour', 'initial_assigned']
Day0 categorical: ['initial_priority', 'initial_issue_type']
Day7 numerical: ['created_weekday', 'created_hour', 'initial_assigned', 'changes_7d', 'status_changes_7d', 'assignee_changes_7d', 'priority_changes_7d', 'comments_7d', 'inactivity_days_at_7d']
Day7 categorical: ['initial_priority', 'initial_issue_type']


In [5]:
out_dir = ROOT / "data" / "processed"
out_dir.mkdir(parents=True, exist_ok=True)

day0.to_parquet(out_dir / "model_day0.parquet", index=False)
day7.to_parquet(out_dir / "model_day7.parquet", index=False)

summarize_missing(day0).to_csv(
    ROOT / "results" / "tables" / "missingness_day0.csv",
    index=False,
)
summarize_missing(day7).to_csv(
    ROOT / "results" / "tables" / "missingness_day7.csv",
    index=False,
)

print("Saved model_day0.parquet")
print("Saved model_day7.parquet")

Saved model_day0.parquet
Saved model_day7.parquet


### Ghi chú phương pháp

`initial_priority` và `initial_issue_type` được tái dựng bằng changelog khi field từng thay đổi.  
Nếu changelog không đầy đủ ở một repository, phải ghi đây là hạn chế dữ liệu.

Các cột `snapshot_*` được lưu để audit/sensitivity analysis nhưng mặc định không đi vào strict Day-0 model.